![Egeria Logo](https://raw.githubusercontent.com/odpi/egeria/main/assets/img/ODPi_Egeria_Logo_color.png)

### Coco Pharmaceuticals Lab
----

# Going CFC-Free: replacing the New York CFC refrigeration unit

## Introduction

When [Tom Tally](https://egeria-project.org/practices/coco-pharmaceuticals/personas/tom-tally/) loaded the company's refrigeration units into Egeria (see [load-refrigeration-units.ipynb](load-refrigeration-units.ipynb)), every site had at least one unit that still used a chlorofluorocarbon (CFC) refrigerant.  The budget to replace them was approved, and the race was on to see which site would be the first to be CFC-free.

The New York Lab won.  [Zach Now](https://egeria-project.org/practices/coco-pharmaceuticals/personas), its founder and leader, had suggested the [CFC-Free Initiative](https://egeria-project.org/practices/coco-pharmaceuticals/scenarios/sustainability-initiative/overview/#the-cfc-free-initiative) in the first place.  Forewarned, Zach had already ordered a replacement for the lab's only CFC unit - **REF-NYC-009**, a legacy -40°C freezer using R-12 that stores chemical reagents and solvents.

This notebook records the replacement in Egeria in three steps:

1. Zach registers the new refrigeration unit, **REF-NYC-010**, and links it to the New York Site.
2. Tom retires the old unit.  Its *deploymentStatus* is set to *DISABLED* and then it is archived.
3. Tom checks that the New York Site no longer has any active CFC units.

This notebook needs [load-refrigeration-units.ipynb](load-refrigeration-units.ipynb) to have been run first.

----

## Setting up

The first code cell sets up the connection to Egeria.  If you are using the standard quickstart environment then there is no need to change these values.

Zach uses the [Asset Maker](https://egeria-project.org/services/omvs/asset-maker/overview/) client to create the new unit and the [Location Arena](https://egeria-project.org/services/omvs/location-arena/overview/) client to link it to the New York Site.  Tom uses the Asset Maker client to update the old unit, and the [Metadata Expert](https://egeria-project.org/services/omvs/metadata-expert/overview/) client to archive it.  The [Classification Explorer](https://egeria-project.org/services/omvs/classification-explorer/overview/) client looks up elements by their unique name.

----

In [ ]:
# Initialize pyegeria

import os
view_server = os.environ.get("EGERIA_VIEW_SERVER","qs-view-server")
url = os.environ.get("EGERIA_VIEW_SERVER_URL","https://host.docker.internal:9443")
user_pwd = os.environ.get("EGERIA_USER_PASSWORD")

zachs_user_id = "zachnow"
toms_user_id = "tomtally"

print("\n")
print("Accessing view server " + view_server + " on platform " + url + " for users " + zachs_user_id + " and " + toms_user_id)
print("\n")

from IPython.display import display, Markdown
import json

from pyegeria import AssetMaker, LocationArena, MetadataExpert, ClassificationExplorer
from pyegeria.core._globals import NO_ELEMENTS_FOUND

# Zach registers the new unit for the New York Lab
zachs_asset_client = AssetMaker(view_server, url, zachs_user_id, user_pwd)
zachs_asset_client.create_egeria_bearer_token()

zachs_location_client = LocationArena(view_server, url, zachs_user_id, user_pwd)
zachs_location_client.create_egeria_bearer_token()

# Tom retires the old unit
toms_asset_client = AssetMaker(view_server, url, toms_user_id, user_pwd)
toms_asset_client.create_egeria_bearer_token()

toms_expert_client = MetadataExpert(view_server, url, toms_user_id, user_pwd)
toms_expert_client.create_egeria_bearer_token()

# ... and both look up elements by their unique name
toms_explorer_client = ClassificationExplorer(view_server, url, toms_user_id, user_pwd)
toms_explorer_client.create_egeria_bearer_token()

print("Connected to Egeria")

----

The first function below looks up an element by its unique name.

Archived elements are not returned by normal queries.  Egeria also renames an element when it archives it, adding `_archivedOn_` and the date to its unique name, so that a new element can reuse the original name.  The second function finds an archived refrigeration unit by searching for names that start with the original name followed by `_archivedOn_`.  Setting `forLineage` in the request includes archived elements in the search.

----

In [ ]:
def get_guid(qualified_name: str) -> str | None:
    guid = toms_explorer_client.get_element_guid_by_unique_name(qualified_name)
    if guid is None or guid == NO_ELEMENTS_FOUND:
        return None
    return guid


def get_archived_unit_guid(qualified_name: str) -> str | None:
    body = {
        "class": "SearchStringRequestBody",
        "searchString": qualified_name + "_archivedOn_",
        "startsWith": True,
        "ignoreCase": False,
        "metadataElementTypeName": "RefrigerationUnit",
        "forLineage": True,
    }
    archived_units = toms_asset_client.find_assets(body=body, output_format="JSON")
    if isinstance(archived_units, list) and archived_units:
        return archived_units[0]["elementHeader"]["guid"]
    return None


old_unit_qualified_name = "RefrigerationUnit::REF-NYC-009"
new_unit_qualified_name = "RefrigerationUnit::REF-NYC-010"
new_york_site_qualified_name = "Site::NY-SITE"

new_york_site_guid = get_guid(new_york_site_qualified_name)
old_unit_guid = get_guid(old_unit_qualified_name)
archived_old_unit_guid = get_archived_unit_guid(old_unit_qualified_name)

print(f"{new_york_site_qualified_name} is {new_york_site_guid}")
if old_unit_guid:
    print(f"{old_unit_qualified_name} is {old_unit_guid}")
elif archived_old_unit_guid:
    print(f"{old_unit_qualified_name} was archived - its guid is {archived_old_unit_guid}")
else:
    print(f"{old_unit_qualified_name} not found - has load-refrigeration-units.ipynb been run?")

----

## Step 1: Registering the new refrigeration unit

The new freezer uses R-290 (propane), a natural refrigerant with no ozone depletion potential and a very low global warming potential.  Zach registers it in the same way as the units loaded from Tom's spreadsheet: it is its own anchor, its properties are passed in the *InfrastructureProperties* class with the attributes from the new types in `extendedProperties`, and it is linked to its site with the *KnownLocation* relationship.  It is in use, so its *deploymentStatus* is *ACTIVE*.

The `additionalProperties` record which unit it replaces.

If the new unit is already registered, it is not created again.

----

In [ ]:
new_unit_guid = get_guid(new_unit_qualified_name)

if new_unit_guid:
    print(f"{new_unit_qualified_name} already exists with guid {new_unit_guid}")
else:
    body = {
        "class": "NewElementRequestBody",
        "isOwnAnchor": True,
        "properties": {
            "class": "InfrastructureProperties",
            "typeName": "RefrigerationUnit",
            "qualifiedName": new_unit_qualified_name,
            "displayName": "REF-NYC-010",
            "description": "Panasonic MDF-U5412H-PE: -40°C freezer for chemical reagent and solvent storage - New York lab",
            "deploymentStatus": "ACTIVE",
            "additionalProperties": {
                "replaces": old_unit_qualified_name,
            },
            "extendedProperties": {
                "make": "Panasonic",
                "model": "MDF-U5412H-PE",
                "purchaseDate": "2026-10-12",
                "usedFor": "-40°C freezer for chemical reagent and solvent storage - New York lab",
                "refrigerantType": "R-290",
                "refrigerantVolume": "0.15 kg",
                "refrigerantPercentageLossPerYear": "0.1",
            },
        },
    }
    new_unit_guid = zachs_asset_client.create_asset(asset_type=["InfrastructureProperties"], body=body)
    zachs_location_client.link_known_location(new_unit_guid, new_york_site_guid)
    print(f"{new_unit_qualified_name} created with guid {new_unit_guid} and linked to {new_york_site_qualified_name}")

----

## Step 2: Retiring the old CFC unit

With the new freezer running and the reagents moved across, the old unit is switched off and its refrigerant recovered for safe disposal.  Tom records this in two steps:

1. The *deploymentStatus* of the old unit is set to *DISABLED* - it is no longer in use.  This is a *merge update*, so its other properties are unchanged.
2. The old unit is [archived](https://egeria-project.org/concepts/memento).  Archiving adds the *Memento* classification to the element.  It is no longer returned by normal queries, but it is kept in the repository so that the history of the unit - and the evidence of the initiative's progress - is not lost.  The archive request records the process that archived it and properties that explain why.  Egeria adds the date and time of the archive.

If the old unit has already been archived, this step is skipped.

----

In [ ]:
if archived_old_unit_guid:
    print(f"{old_unit_qualified_name} has already been archived")
elif old_unit_guid:
    update_body = {
        "class": "UpdateElementRequestBody",
        "mergeUpdate": True,
        "properties": {
            "class": "InfrastructureProperties",
            "typeName": "RefrigerationUnit",
            "deploymentStatus": "DISABLED",
        },
    }
    toms_asset_client.update_asset(old_unit_guid, update_body)
    print(f"{old_unit_qualified_name} is now DISABLED")

    # archiveDate is left out: Egeria uses the current time, and pyegeria 6.2.10 cannot serialise a
    # supplied archiveDate (it is converted to a datetime that is not dumped in JSON mode).
    archive_body = {
        "class": "DeleteElementRequestBody",
        "archiveProcess": "CFC-Free Initiative",
        "archiveProperties": {
            "reason": "Unit used the CFC refrigerant R-12.  Refrigerant recovered for safe disposal.",
            "replacedBy": new_unit_qualified_name,
        },
    }
    toms_expert_client.archive_metadata_element(old_unit_guid, archive_body)
    archived_old_unit_guid = old_unit_guid
    print(f"{old_unit_qualified_name} has been archived")

----

The old unit can still be retrieved by setting `forLineage`.  It shows its new unique name and the *DISABLED* deployment status, and the *Memento* classification records the details of the archive.

----

In [ ]:
old_unit = toms_expert_client.get_metadata_element_by_guid(archived_old_unit_guid, body={"class": "GetRequestBody", "forLineage": True})

if isinstance(old_unit, dict):
    old_unit_properties = old_unit.get("elementProperties", {}).get("propertiesAsStrings", {})
    print("qualifiedName:    " + old_unit_properties.get("qualifiedName", ""))
    print("deploymentStatus: " + old_unit_properties.get("deploymentStatus", ""))
    print("refrigerantType:  " + old_unit_properties.get("refrigerantType", ""))
    for classification in old_unit.get("classifications", []):
        if classification.get("classificationName") == "Memento":
            print("\nMemento classification:")
            print(json.dumps(classification.get("classificationProperties", {}).get("propertiesAsStrings"), indent=2))
else:
    print(old_unit)

----

## Step 3: Checking that the New York Site is CFC-free

Finally, Tom lists the active refrigeration units at the New York Site.  The archived unit is no longer returned, and none of the units that remain use a CFC refrigerant.

----

In [ ]:
cfc_refrigerants = ["R-11", "R-12", "R-500", "R-502"]

new_york_units = toms_asset_client.find_assets("RefrigerationUnit::REF-NYC-", starts_with=True,
                                               metadata_element_type="RefrigerationUnit", output_format="JSON")

report = "| Unit | Make | Model | Refrigerant | Deployment status |\n|---|---|---|---|---|\n"
active_cfc_units = []
for unit in sorted(new_york_units if isinstance(new_york_units, list) else [], key=lambda u: u["properties"]["qualifiedName"]):
    properties = unit["properties"]
    extended_properties = properties.get("extendedProperties", {})
    refrigerant_type = extended_properties.get("refrigerantType")
    if refrigerant_type in cfc_refrigerants:
        active_cfc_units.append(properties.get("displayName"))
    report += (f"| {properties.get('displayName')} | {extended_properties.get('make')} | {extended_properties.get('model')} "
               f"| {refrigerant_type} | {properties.get('deploymentStatus')} |\n")

display(Markdown(report))

if active_cfc_units:
    print("The New York Site still has CFC units: " + ", ".join(active_cfc_units))
else:
    print("The New York Site is CFC-free!")

----

## Next steps

With its last CFC unit retired, the New York Lab is the first site to be CFC-free.  Tom can now award Zach the CFC-Free certificate - a [certification](https://egeria-project.org/types/4/0482-Certifications) attached to the New York Site.  The other sites follow over the next few months, until the whole company is CFC-free.

----
License: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/),
Copyright Contributors to the ODPi Egeria project.